graph based risk aware aircraft colision avoidance using value iteration and policy iteration 

Graph-Risk MDP: Dynamic Programming Based Aircraft Collision Avoidance using OpenSky Encounter Data

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter,defaultdict
import os
import re
import json

: 

In [ ]:
df=pd.read_csv('opensky_collision_avoidance_dataset.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.columns 

In [ ]:
df['mdp_state'].nunique()

In [ ]:
df['recommended_action'].value_counts()

lets make transitio 

In [ ]:
df=df.sort_values(['own_icao24','intruder_icao24','time']).reset_index(drop=True)
df['next_state']=df.groupby(['own_icao24','intruder_icao24'])['mdp_state'].shift(-1)
df=df.dropna(subset=['next_state']).copy()

states=sorted(df['mdp_state'].unique())
actions=sorted(df['recommended_action'].unique())
print(f'number of states: {len(states)}')
print(f'number of actions: {len(actions)}')

MDP probability 

In [ ]:
transition_counts=defaultdict(Counter)
reward_values=defaultdict(list)

for _,row in df.iterrows():
    s=row['mdp_state']
    a=row['recommended_action']
    s_next=row['next_state']
    r=row['reward']
    
    transition_counts[(s,a)][s_next]+=1
    reward_values[(s,a)].append(r)
    
p={}
r={}
for key,counter in transition_counts.items():
    total=sum(counter.values())
    p[key]={s_next:count/total for s_next,count in counter.items()}
    
    
for key,rewards in reward_values.items():
    r[key]=np.mean(rewards)

value iteration 

policy dictionary 

In [ ]:
gamma=0.99 
theta = 1e-6 
max_iter=1000

v={s:0 for s in states}
for it in range(max_iter):
    delta= 0 
    for s in states:
        q_values=[]
        
        for a in actions:
            if (s,a) not in p :
                q_values.append(-1e9)
                continue
            
            q=0 
            for ns ,prob in p[(s,a)].items():
                reward= r.get((s,a,ns),0)
                q+=prob*(reward + gamma * v[ns])
            q_values.append(q)
            
        best_value=max(q_values)
        delta=max(delta,abs(best_value-v[s]))
        v[s]=best_value
        
    if delta < theta:
        print(f'Value iteration converged after {it+1} iterations.')
        break
value_policy={}
for s in states:
    best_a = None 
    best_q = -1e9
    for a in actions:
        if (s,a) not in p:
            continue 
        q=0 
        for ns,prob in p[(s,a)].items():
            reward=r.get((s,a,ns),0)
            q+=prob*(reward + gamma * v[ns])
        if q > best_q:
            best_q=q
            best_a=a
    value_policy[s]=best_a
value_policy 

policy iteration 

In [ ]:
policy = {s: np.random.choice(actions) for s in states}

def policy_evaluation(policy, gamma=0.90, theta=1e-6):
    V = {s: 0.0 for s in states}

    while True:
        delta = 0

        for s in states:
            a = policy[s]

            if (s, a) not in P:
                continue

            old_v = V[s]
            new_v = 0

            for ns, prob in P[(s, a)].items():
                reward = R.get((s, a, ns), 0)
                new_v += prob * (reward + gamma * V[ns])

            V[s] = new_v
            delta = max(delta, abs(old_v - new_v))

        if delta < theta:
            break

    return V


def policy_improvement(V, policy):
    stable = True
    new_policy = policy.copy()

    for s in states:
        old_action = policy[s]

        best_a = None
        best_q = -1e9

        for a in actions:
            if (s, a) not in P:
                continue

            q = 0
            for ns, prob in P[(s, a)].items():
                reward = R.get((s, a, ns), 0)
                q += prob * (reward + gamma * V[ns])

            if q > best_q:
                best_q = q
                best_a = a

        if best_a is not None:
            new_policy[s] = best_a

        if new_policy[s] != old_action:
            stable = False

    return new_policy, stable


for it in range(100):
    V_pi = policy_evaluation(policy)
    new_policy, stable = policy_improvement(V_pi, policy)

    policy = new_policy

    if stable:
        print("Policy Iteration converged at iteration:", it)
        break

policy_iteration_policy = policy
policy_iteration_policy